In [ ]:
import astropy.constants as const
import astropy.units as u
import corner
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import yaml
from astropy.cosmology import Planck18
from scipy.stats import poisson

import zombie_rate as rate
import PTA as pta_utils
import smbhbs
import style.fig_style as fig_style
import utils
from binary_pop_model import analytic_hc2, getdndlogMcr, smbhb_density

# Load configs

In [ ]:
# Load the population models config file
config_file = "./config/fig_config.yaml"

# Load the different configs
run_config, _ = utils.load_configs(config_file)

config_file = f"./config/binary_models/pop_models_1.2.yaml"
with open(config_file, 'r') as file:
    pop_config = yaml.safe_load(file)

# Randomness of the extrinsic zombie parameters
rng = np.random.default_rng(seed=run_config['seed'])

# The Earth term orbital frequency threshold (from NANOGrav upper bound)
fth = run_config['fth'] * u.Hz

# Load population models

In [ ]:
### Define binary population models
pop_models = pop_config['models']
Nmodels = len(pop_models)

for i, pop_model in enumerate(pop_models):
    # Convert n0_dot to correct dimensions
    pop_model['n0_dot'] /= (u.Gyr * u.Mpc ** 3)

## Rescale $\dot{n}_0$ (if needed) to have correct $h_{\rm c}(f=1/yr)$

In [ ]:
## Derive their respective n0_dot to have hc(fyr) = 3e-15
fyr = (1 / u.yr).to(u.Hz)

bounds = pop_config['bounds']

for i, pop_model in enumerate(pop_models):
    # Compute the reference characteristic strain at f = 1/yr for this population model
    hc_ref = np.sqrt(analytic_hc2(f=fyr, 
                                  smbhb_density=smbhb_density, 
                                  model=pop_model, 
                                  xi_I_bounds=bounds))
    
    # Check that the model is compatible with PTA observations
    if not np.allclose(hc_ref, run_config['hc_ref'], rtol=5e-3):
        # Compute the scaling factor for n0_dot to obtain hc(1/yr) = 2e-15 (compatible with PTA results)
        n0dot_factor = (run_config['hc_ref'] ** 2 / hc_ref ** 2).to(u.dimensionless_unscaled)

        # Update the n0dot of the model
        pop_model['n0_dot'] *= n0dot_factor

        print(f"Model {i+1} n·_0 = {pop_model['n0_dot']:.2e} \
                in log_10 => {np.log10(pop_model['n0_dot'].value)}")

# Paper plots

## Comoving merger density

In [ ]:
log10_Mcr_s = np.linspace(bounds['log10_Mcr_min'], bounds['log10_Mcr_max'], 50)
lss = ['solid', '--', 'dotted']
lws = [3, 3, 3.5]

for ip, pop_model in enumerate(pop_models):
    dndlogMcr = getdndlogMcr(log10_Mcr=log10_Mcr_s, pop_model=pop_model,
                             z_min=bounds['z_min'], z_max=bounds['z_max'])
    
    plt.semilogy(log10_Mcr_s, dndlogMcr, lw=lws[ip], ls=lss[ip], label=f"Model {pop_model['name']}")
plt.xlabel(r"$\log \mathcal{M}_{\rm c, r}$", fontsize=15)
plt.ylabel(r"$\frac{d n}{d\log \mathcal{M}_{\rm c, r}}$ [Mpc$^{-3}$ dex$^{-1}$]", fontsize=15)
plt.xlim([bounds['log10_Mcr_min'], bounds['log10_Mcr_max']])
plt.ylim([1e-10, 1])
plt.legend(fontsize=13.5)
plt.savefig('./plots/merger_density_Mcr.png')
plt.savefig('./plots/merger_density_Mcr.pdf')
plt.show()

## $f_{\rm orb}^{(P),a}$

In [ ]:
# Range of time delay and observer frame chirp mass
log_Dtau_tot_s = np.linspace(0, 4.5, 150)
Dtau_tot_s = 10 ** log_Dtau_tot_s * u.yr

log10_Mc_s = np.linspace(7, 11, 100)
Mc_s = 10 ** log10_Mc_s * const.M_sun

def getforbPa_zombie_Dtot(Mc: float,
                          Delta_tau_tot: float,
                          q: float = 1) -> float:
    """
    Compute the orbital frequency at the pulsar term of pulsar a located at p = c / psr_T * psr_pos
    of a binary located at binary_pos.

    Args:
        Mc (float): Chirp mass of the binary (observer frame)
        Delta_tau_tot (float): Delay between coalescence and starting time of the pulsar term
        q (float): Mass ratio of the binary

    Returns:
        float: Orbital frequency of pulsar a pulsar term
    """

    # Compute total mass of the binary
    Mtot = smbhbs.getMtot(Mc=Mc, q=q)

    # Compute ISCO frequency
    forb_ISCO = smbhbs.getforb_ISCO(Mtot=Mtot)

    # Compute characteristic coalescence time
    Tc = smbhbs.getCoalescenceTime(Mc=Mc, forb=forb_ISCO)

    return (
        forb_ISCO * (
            1 + (Delta_tau_tot / Tc).to(u.dimensionless_unscaled)
        ) ** (-3 / 8)
    )

forbP_s = getforbPa_zombie_Dtot(Mc=Mc_s[:,None], 
                                Delta_tau_tot=Dtau_tot_s[None,:],
                                q=1) # NOTE: we assume q=1 here

plt.figure(figsize=(6, 4))
im = plt.imshow(np.log10(forbP_s.to(u.Hz).value), aspect='auto', origin='lower',
                extent=[log10_Mc_s[0], log10_Mc_s[-1],
                        log_Dtau_tot_s[0], log_Dtau_tot_s[-1]],
                cmap='cubehelix')
plt.colorbar(im, label=r'$\log f_{\mathrm{orb}}^{\mathrm{(P)}} / \mathrm{Hz}$')

# Show decade frequencies for guidance
levels = [-9, -8, -7]
CS = plt.contour(np.log10(forbP_s.to(u.Hz).value), levels=levels, 
                 extent=[log10_Mc_s[0], log10_Mc_s[-1], 
                         log_Dtau_tot_s[0], log_Dtau_tot_s[-1]],
                 origin='lower', colors='white', linewidths=1.5, alpha=1)

# Add inline labels
fmt = {levels[0]: '1 nHz', levels[1]: '10 nHz', levels[2]: '100 nHz'}
plt.clabel(CS, inline=True, fmt=fmt, fontsize=12, colors='white')

plt.xlabel(r'$\log (\mathcal{M}_{\mathrm{c}} / M_{\odot})$')
plt.ylabel(r'$\log \left[ (\tau_{a}  - \tau_{\mathrm{c}}) / \mathrm{yr} \right]$')
plt.tight_layout()
plt.savefig(f'./plots/forbP_grid.pdf')
plt.savefig(f'./plots/forbP_grid.png')
plt.show()

## Detection efficiencies

In [ ]:
runs = ['EPTA-noRN-logz', 'IPTA-noRN-logz', 'SKA-130-opt-noRN-logz']

fig, axs = plt.subplots(1, 3, figsize=(18, 4))

for ir, run in enumerate(runs):
    # Load run config
    config_file = f"./results/{run}/config.yaml"
    with open(config_file, 'r') as file:
        run_config = yaml.safe_load(file)

    # Load efficiency grid
    efficiency_grid = np.load(f"./results/{run}/efficiency-grid.npz")

    # Load elements
    efficiency = efficiency_grid['efficiency']

    log10_Mcr_mids = efficiency_grid['log10_Mcr_mids']
    z_mids = efficiency_grid['z_mids']
    # If logz is not there it is False
    logz = efficiency_grid.get('logz', False)
    
    # Create the meshgrid
    X, Y = np.meshgrid(log10_Mcr_mids, z_mids)
    
    # Convert to log10 of efficiency
    efficiency = np.where(efficiency > 0, 
                          np.log10(efficiency), 
                          np.nan)

    pcm = axs[ir].pcolormesh(X, Y, efficiency.T, cmap='cubehelix')
    if logz:
        axs[ir].set_yscale('log')

    axs[ir].set_xlabel(r'$\log \mathcal{M}_{\mathrm{c,r}} / M_{\odot}$', fontsize=15)
    axs[ir].set_ylabel(r'$z$', fontsize=15)
    axs[ir].set_title(f"{run.split('-')[0]}", fontsize=18)
cbar = fig.colorbar(pcm, ax=axs.ravel().tolist())
cbar.ax.tick_params(labelsize=14)
cbar.set_label(label=fr'$\log$ P$(\mathrm{{SNR}} > {run_config['SNR_thresh']})$', size=16)

plt.savefig("./plots/log-efficiencies.pdf", bbox_inches='tight')
plt.savefig("./plots/log-efficiencies.png", bbox_inches='tight')
plt.show()

## Average zombie numbers

In [ ]:
runs = ['EPTA-noRN-logz', 'IPTA-noRN-logz', 'SKA-130-opt-noRN-logz']
colors = ['tomato', 'midnightblue', 'orange']
markers = ['x', 'o', '^']

Nmodels = len(pop_models)
x = np.arange(Nmodels)

fig, ax = plt.subplots(figsize=(6,4))

for i_pta, run in enumerate(runs):
    # Load efficiency grid
    efficiency_grid = np.load(f"./results/{run}/efficiency-grid.npz")

    # Load PTA config
    config_file = f"./results/{run}/PTA.yaml"
    with open(config_file, 'r') as file:
            PTA = yaml.safe_load(file)
            PTA['Ta_max'] = PTA['Ta_max'] * u.yr

    # Load elements
    efficiency = efficiency_grid['efficiency']
    log10_Mcr_mids = efficiency_grid['log10_Mcr_mids']
    z_mids = efficiency_grid['z_mids']
    if efficiency_grid.get('logz', False):
        z_edges = efficiency_grid['z_edges']
        assert np.allclose(np.diff(np.log10(z_edges)), np.diff(np.log10(z_edges))[0])
        Dlogz = np.diff(np.log10(z_edges))[0]
    else:
        Dlogz = None

    Nbar_s = np.zeros(Nmodels)
    for ip, pop_model in enumerate(pop_models):
        Nbar_s[ip] = rate.getNbarZombies(pop_model=pop_model, 
                                         efficiency=efficiency,
                                         log10_Mcr_mids=log10_Mcr_mids,
                                         z_mids=z_mids,
                                         Ta_max=PTA['Ta_max'],
                                         fth=fth,
                                         Dlogz=Dlogz)
        

    # Compute probability of at least one event
    prob_det_s = 1 - poisson.pmf(k=0, mu=Nbar_s) # poisson.cdf(k=0, mu=Nbar_s)

    ax.scatter(x, prob_det_s, 
               marker=markers[i_pta], linewidths=4,
               color=colors[i_pta],
               label=f"{run.split('-')[0]}")

labels = [f"Model M{i+1}" for i in range(Nmodels)]
ax.set_xticks(x, labels, rotation=45, fontsize=15)
ax.set_ylabel(r"P$(N_{\mathrm{z}} > 0)$", fontsize=15)
ax.set_yscale('log')

plt.legend(fontsize=13)
plt.tight_layout()
plt.savefig(f"./plots/Nz-models-PTAs-logz.pdf")
plt.savefig(f"./plots/Nz-models-PTAs-logz.png")
plt.show()

## Zombie properties

In [ ]:
runs = ['SKA-130-opt-noRN-logz']
PTA_files = ['SKA.npy'] # File containing the positions and distances of the PTA pulsars

# Number of zombies to draw from distribution
N_zombies = 20_000
N_per_chunk = 500
N_chunk = N_zombies // N_per_chunk

# Get number of population models to test
Nmodels = len(pop_models)

# Fix randomness
seed = 42
rng = np.random.default_rng(seed=seed)

for i_pta, run in enumerate(runs):
        # Load efficiency grid
        efficiency_grid = np.load(f"./results/{run}/efficiency-grid.npz")

        # Load PTA config
        config_file = f"./results/{run}/PTA.yaml"
        with open(config_file, 'r') as file:
                PTA = yaml.safe_load(file)

        # Load run config
        config_file = f"./results/{run}/config.yaml"
        with open(config_file, 'r') as file:
                run_config = yaml.safe_load(file)

        # Setup the pta object
        pta_utils.setup_pta(run_config=run_config, PTA=PTA)

        # Load elements
        efficiency = efficiency_grid['efficiency']
        log10_Mcr_mids = efficiency_grid['log10_Mcr_mids']
        z_mids = efficiency_grid['z_mids']

        # Construct mesh
        grids = [log10_Mcr_mids, z_mids]
        mesh = np.meshgrid(*grids, indexing='ij')

        # Extract redshift and log_Mcr meshs
        log_Mcr_s, z_s = mesh[0], mesh[1]

        # Sample
        pts = np.stack([m.ravel() for m in mesh], axis=1)

        # Store samples (z, Mcr, tauc, cos_theta, phi, cos_inc, avg_forP, Npsr, SNR)
        prop_samples = np.zeros((N_zombies, 9))

        for pop_model in pop_models:
                # Construct the unormalized density over Mcr and z grid
                pop_density = smbhb_density(log10_M=log_Mcr_s,
                                            z=z_s,
                                            **pop_model)
                
                # Weight the SMBHB density by the detection efficiency
                unn_density = (pop_density *
                               Planck18.luminosity_distance(z_s).to(u.Mpc).value ** 2 /
                               (1 + z_s) ** 2 *
                               efficiency)
                
                if efficiency_grid.get('logz', False):
                        print("Log-z spacing used, need to multiply by z!")
                        unn_density *= z_s

                # Get samples weighted by efficiency
                samples = utils.sample_from_grid(mesh, unn_density,
                                                 N_samples=N_zombies,
                                                 rng=rng)

                # For each intrinsic sample, generate extrinsic parameters and compute SNR
                log_Mcr_samples, z_samples = samples[:,0], samples[:,1]

                for k in range(N_chunk):
                        # Sample extrinsic parameters for this chunk
                        SNRs, pars = rate.sampleZombiePars_jax(z=z_samples[k*N_per_chunk:(k+1)*N_per_chunk],
                                                        log10_Mcr=log_Mcr_samples[k*N_per_chunk:(k+1)*N_per_chunk],
                                                        q=np.atleast_1d(1),
                                                        PTA=PTA,
                                                        fth=run_config['fth'] * u.Hz,
                                                        N_zombies=N_per_chunk,
                                                        rng=rng)
                        
                        # Extract extrinsic pars
                        Delta_tauc = pars['tauc'].to(u.kyr)
                        cos_theta = pars['cos_theta']
                        phi = pars['phi']
                        cos_inc = pars['cos_inc']
                        log10_fgwP_s = pars['log10_fgwP_s'] # (N_zombies, Npsr)

                        # Identify the valid zombie-pulsar pairs
                        PT_valid_pair = ~np.isnan(log10_fgwP_s)

                        # Get number of pulsars seeing each zombie 
                        N_contrib_psrs = np.sum(PT_valid_pair, axis=-1)

                        # Compute average log_forb_P
                        # NOTE: here, we compute log <fgw_P>, not <log_fgwP>
                        fgwP_s = 10 ** log10_fgwP_s * u.Hz
                        avg_fgwP_s = utils.getAvgOverValids(fgwP_s, 
                                                            N_contrib_psrs, 
                                                            PT_valid_pair).to(u.Hz).value

                        # Fill the prop_samples array
                        prop_samples[k*N_per_chunk:(k+1)*N_per_chunk,:] = np.array([np.log10(z_samples[k*N_per_chunk:(k+1)*N_per_chunk]), 
                                                                                log_Mcr_samples[k*N_per_chunk:(k+1)*N_per_chunk],
                                                                                Delta_tauc,
                                                                                cos_theta,
                                                                                phi,
                                                                                cos_inc,
                                                                                avg_fgwP_s,
                                                                                N_contrib_psrs,
                                                                                SNRs]
                        ).T

                ### Plot the corner of prop_samples
                # Select columns to show
                to_show = [0, 1, 2, 6, 7, 8]

                # Select only zombie with at least one pulsar
                N_contrib_psrs_all = prop_samples[:,7]
                mask = np.where(N_contrib_psrs_all > 0)[0]
                valid_samples = prop_samples[mask,:]

                # Convert fgw to log_fgw
                valid_samples[:,6] = np.log10(valid_samples[:,6])

                ### Select SNR > 3 only
                bright_mask = np.where(valid_samples[:,-1] >= 3)[0]
                bright_zombies = valid_samples[bright_mask,:]

                # Convert to log10 SNR
                bright_zombies[:,-1] = np.log10(bright_zombies[:,-1])

                weights = np.ones(bright_zombies.shape[0]) / bright_zombies.shape[0]
                fig = corner.corner(bright_zombies[:,to_show], 
                                    labels=[r"$\log z$",
                                            r"$\log \mathcal{M}_{\mathrm{c, r}} / M_\odot$",
                                            r"$\tau_{\mathrm{c}}$ [kyr]",
                                            r"$\log \: \langle f_{\mathrm{GW}}^{\mathrm{(P)}} \rangle$",
                                            r"$N_{\mathrm{psr}}$",
                                            r"$\log$ SNR"],
                        smooth=True, plot_datapoints=False,
                        levels=(0.393, 0.864), # 1 and 2 sigmas
                        fill_contours=True,
                        weights=weights,
                        quantiles=[.05, .5, .95],
                        title_quantiles=[.05, .5, .95],
                        title_fmt=".2f",
                        show_titles=True, 
                        color='tomato',
                        smooth1d=True)

                ### Add legend
                patch_bright = mpatches.Patch(color='tomato', label=r'SNR $> 3$', alpha=1)
                fig.legend(
                        handles=[patch_bright],
                        loc=(0.7, 0.8), # Coordinates in figure fraction (x, y)
                        fontsize=24                
                )

                # Save the figure
                plt.savefig(f'./plots/zombie-props-{run.split('-')[0]}-pop-{pop_model['name']}.pdf')
                plt.savefig(f'./plots/zombie-props-{run.split('-')[0]}-pop-{pop_model['name']}.png')
                plt.show()
